# Reading a correlation matrix during EDA_Data Wrangling_**Build one heatmap of how your columns move together — then read off the predictors and the redundant twins.**A correlation matrix is a square grid: one row and one column per numeric feature, and the       cell at $(i,j)$ is the correlation between feature $i$ and feature $j$. The diagonal is always $1$       (every column correlates perfectly with itself), and the grid is symmetric (cell $(i,j)$ equals cell       $(j,i)$), so you really only read one triangle.       Paint each cell by its value &mdash; bright for strongly positive, dark for strongly negative,       flat for near-zero &mdash; and the grid becomes a heatmap you can scan in seconds. Two things       jump out: the target's row (which features track what you want to predict) and the bright       off-diagonal blocks (groups of features that are really telling you the same thing).---This notebook builds and reads a correlation matrix one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — pandas + seabornWe'll build a correlation matrix on the `load_wine` table and then *read* it the way you would during EDA. We do it in four steps: (1) compute the matrix, (2) draw the heatmap, (3) rank features by their correlation with the target, and (4) flag redundant near-twin pairs and act on them.

### Step 1 — Build the correlation matrix`df.corr()` returns a square table: one row and column per numeric feature, with each cell holding the Pearson correlation between that pair. The diagonal is always 1 (every column correlates perfectly with itself) and the grid is symmetric. Pearson is the default; `method="spearman"` would instead capture monotonic (ranked) trends.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
df = wine.frame                       # 13 continuous features + 'target' (3 classes)

# Build the correlation matrix (Pearson is the default).
corr = df.corr(method="pearson")      # use method="spearman" for monotonic/ranked trends

### Step 2 — Draw the heatmapPainting each cell by its value turns the grid into a picture you can scan in seconds: warm cells are strongly positive, cool cells strongly negative, pale cells near zero. `center=0` with `vmin=-1, vmax=1` pins the colour scale so the colours are comparable across the whole matrix.

In [ ]:
# Draw the heatmap — the chart every EDA notebook shows.
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"label": "Pearson r"})
plt.title("Correlation matrix — load_wine")
plt.tight_layout()
# plt.show()

### Step 3 — Read it for predictorsThe target's row tells you which features track what you want to predict. We pull that row, drop its own diagonal entry, take absolute values (a strong negative correlation is just as useful as a strong positive one), and sort to surface the top candidate predictors.

In [ ]:
# Read it for PREDICTORS: sort features by |corr with the target|.
target_corr = corr["target"].drop("target")          # the target's row, minus the diagonal
ranked = target_corr.abs().sort_values(ascending=False)

print("Features by |correlation with target|:")
print(ranked.round(3).head(6))
# flavanoids 0.847, od280/... 0.788, total_phenols 0.719, proline 0.634, hue 0.617, ...

### Step 4 — Read it for redundancy, then actBright off-diagonal cells flag features that carry nearly the same information. We look only at the upper triangle (the matrix is symmetric, so the lower half is a mirror), keep pairs above a threshold, and then drop the weaker member of each redundant pair before a linear model — keeping `flavanoids` over its twin `total_phenols`.

In [ ]:
# Read it for REDUNDANCY: flag highly inter-correlated feature pairs.
feat_corr = corr.drop(index="target", columns="target")            # features only
upper = feat_corr.where(np.triu(np.ones(feat_corr.shape, dtype=bool), k=1))  # upper triangle

THRESH = 0.8
strong = upper.abs().stack()                       # (f1, f2) -> |r|
pairs = strong.loc[lambda s: s > THRESH].sort_values(ascending=False)

print("Redundant feature pairs (|r| > %.1f):" % THRESH)
print(pairs.round(3))
# total_phenols / flavanoids 0.865 -> near-twins: drop one, merge, or PCA the group.

# Act on it: drop one column from each redundant pair before a LINEAR model.
to_drop = {"total_phenols"}            # keep flavanoids (stronger target corr), drop its twin
df_model = df.drop(columns=list(to_drop))
print("\nKept", df_model.shape[1] - 1, "features after dropping", to_drop)

## Visualize the data & results_On the real load_wine table, which features correlate most strongly with the wine class (candidate predictors) — and which feature pairs are redundant near-twins you'd flag in the matrix?_

### Step 1 — Rank features by correlation with the targetUsing plain NumPy this time: we append the target as a final column, run `np.corrcoef` across all columns, and read the last column of the result — that holds each feature's correlation with the target. Sorting by absolute value surfaces the top candidate predictors, matching the pandas run above.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine

wine = load_wine()
names = list(wine.feature_names)
X, y = wine.data, wine.target

# Bars: each feature's |Pearson r| with the target, sorted.
Xt = np.column_stack([X, y])             # append target as last column
ft = np.corrcoef(Xt.T)[:-1, -1]          # feature-vs-target correlations
order = np.argsort(np.abs(ft))[::-1][:8]
for i in order:
    print(f"{names[i]:>30s}  |r| = {abs(ft[i]):.3f}")
# flavanoids 0.847, od280/... 0.788, total_phenols 0.719, proline 0.634, ...

### Step 2 — Inspect a small feature-to-feature matrixNow we pick six interesting features and compute the correlation matrix among *them*. The result is a 6×6 grid with 1.0 down the diagonal; the bright off-diagonal cells are the near-twin pairs you would flag for redundancy.

In [ ]:
# Heatmap: Pearson correlation matrix among 6 chosen features.
pick = ["alcohol", "flavanoids", "color_intensity", "hue",
        "od280/od315_of_diluted_wines", "proline"]
idx = [names.index(p) for p in pick]
corr = np.corrcoef(X[:, idx].T)          # 6x6, diagonal = 1.0
print(np.round(corr, 2))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You build df.corr() on 13 numeric features plus the target. The target's row shows flavanoids at r = -0.85 and total_phenols at r = -0.72, and the off-diagonal cell between those two features is r = +0.86. How do you read this, and what do you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look at the target's row first and sort by absolute correlation. — _Both features have large $|r|$ with the target, so both are strong candidate predictors; the negative sign just means they fall as the target class index rises._
- Now look at the off-diagonal cell between the two features. — _$r=+0.86$ means flavanoids and total_phenols are near-twins &mdash; they carry almost the same information (redundancy / multicollinearity)._
- Decide what to keep. — _For a linear model, feeding both inflates and destabilizes the coefficients; keep the stronger predictor (flavanoids), or merge the group with PCA, or use ridge._

**Answer:** Both flavanoids ($|r|=0.85$) and total_phenols ($|r|=0.72$) are strong candidate predictors &mdash; the negative sign is fine, you rank on magnitude. But their mutual $r=0.86$ flags them as redundant near-twins. For a linear model that redundancy is multicollinearity, so don't keep both blindly: keep flavanoids (the stronger one), or collapse the correlated group with PCA, or switch to ridge regression. A tree model tolerates the redundancy better but still gains little from the duplicate.

</details>

**Problem 2.** A feature X plotted against the target looks like a clear smile-shaped (U) curve, yet df.corr() reports r ≈ 0.03 for that cell. A teammate says "near-zero correlation, drop X." Right or wrong?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what the matrix's default measure is. — _df.corr() is Pearson by default, which measures only a straight-line trend._
- Match it to the shape you see. — _A symmetric U has no net linear slope, so Pearson $r\approx 0$ even though the relationship is strong &mdash; this is the Anscombe lesson._
- Reach for a measure that sees the shape. — _Mutual Information (any shape) catches the U; Spearman would also be low because a U is not monotonic, which is itself the clue the link is curved, not absent._

**Answer:** Wrong. The matrix's default is Pearson, which only sees straight lines, so $r\approx 0$ rules out a line, not a relationship. The U-curve is a real nonlinear dependence Pearson is blind to. Keep X: confirm with Mutual Information (it flags the U) and consider a squared term or a nonlinear model. This is exactly why you never read a correlation cell without the scatter &mdash; Anscombe's quartet shows four datasets with the same $r$ and four different shapes.

</details>

**Problem 3.** On 40 rows and 200 features you scan the correlation matrix and the brightest off-diagonal cell is r = 0.61 between two unrelated-seeming columns. Should you trust it as a real relationship?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count how many pairs you searched over. — _200 features give about 20,000 distinct pairs; with only 40 rows, plenty will hit $|r|\approx 0.6$ by pure chance._
- Recognize the multiple-comparisons trap. — _Cherry-picking the single brightest cell out of thousands is selecting on noise &mdash; a spurious correlation._
- Validate instead of trusting. — _Check the relationship on fresh data (or a held-out split), correct for multiple comparisons, and confirm with a scatter before reading anything into it._

**Answer:** Don't trust it yet. With 200 features there are roughly 20,000 pairs, and on only 40 rows a $|r|$ near $0.6$ will appear by chance for some pair &mdash; this is the spurious-correlation / multiple-comparisons trap. Picking the single brightest cell out of thousands is selecting on noise. Fix: verify on a fresh split or new data, apply a multiple-comparison correction, and always look at the scatter before believing a wide-table correlation.

</details>